# Lab 5 — Cleaning and Transforming Data in Pandas

**Course:** Python for AI & Data  
**Analyst:** PaperLane Analytics Team  
**Date:** 2026-01-15

---

## The PaperLane Scenario

PaperLane's raw order data has been cleaned and transformed into an analysis-ready dataset. This notebook documents all cleaning decisions.


---
## Task 1 — Load and First Look


In [ ]:
import pandas as pd
from pathlib import Path

RAW = Path("data") / "raw"
PROCESSED = Path("data") / "processed"

df = pd.read_csv(RAW / "paperlane_orders.csv")

df.head()


In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


---
## Task 2 — Structured Inspection


In [ ]:
df.info()


In [ ]:
print("Missing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
print("status value counts:")
print(df["status"].value_counts())

print("\nregion value counts:")
print(df["region"].value_counts())

print("\nchannel value counts:")
print(df["channel"].value_counts())


### Data Quality Issues Found

After inspection, the following issues were identified:

| Issue | Column | Detail |
|-------|--------|---------|
| Duplicate row | All columns | `order_id` 1007 appears twice |
| Missing values | `unit_price` | 2 rows missing (orders 1004, 1013) |
| Missing values | `region` | 2 rows missing (orders 1008, 1020) |
| Mixed casing | `status` | `Complete`, `complete`, `COMPLETE`, `returned` |
| Leading whitespace | `region` | `" West"` appears as a separate value |
| Trailing whitespace | `region` | `"Midwest "` appears as a separate value |
| Non-standard date format | `order_date` | Two rows use `MM/DD/YYYY` instead of `YYYY-MM-DD` |


---
## Task 3 — Remove Duplicates


In [ ]:
print("Before:", df.shape)
df = df.drop_duplicates()
print("After:", df.shape)


---
## Task 4 — Handle Missing Values


In [ ]:
# Fill missing unit_price with the median
median_price = df["unit_price"].median()
df["unit_price"] = df["unit_price"].fillna(median_price)
print(f"Filled missing unit_price with median: ${median_price:.2f}")

# Fill missing region with 'Unknown'
df["region"] = df["region"].fillna("Unknown")

# Verify
print(df.isna().sum())


### Cleaning Decisions

**`unit_price`:** Filled with the median rather than the mean because the price distribution is right-skewed (Backpack at $39 pulls the mean up). The median is more representative of a typical product price. Dropping these rows was considered but rejected — they contain valid quantity and customer data.

**`region`:** Filled with `"Unknown"` rather than dropping because the orders are otherwise complete. Dropping would remove valid revenue data. The `"Unknown"` label makes these rows visible in any regional grouping rather than silently disappearing.


---
## Task 5 — Standardise Text Columns


In [ ]:
df["status"] = df["status"].str.strip().str.title()
df["region"] = df["region"].str.strip()
df["channel"] = df["channel"].str.strip()

print(df["status"].value_counts())
print()
print(df["region"].value_counts())


---
## Task 6 — Convert the Date Column


In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], dayfirst=False)
print(df["order_date"].dtype)
df["order_date"].head()


---
## Task 7 — Create Derived Columns


In [ ]:
df["revenue"] = df["quantity"] * df["unit_price"]
df["order_month"] = df["order_date"].dt.month

df[["order_date", "order_month", "product", "quantity", "unit_price", "revenue"]].head()


---
## Task 8 — Filter and Sort


In [ ]:
df_complete = df[df["status"] == "Complete"].copy()
df_complete_sorted = df_complete.sort_values("revenue", ascending=False)

print(f"Complete orders: {len(df_complete)} of {len(df)} total")
df_complete_sorted.head()


---
## Task 9 — Save Cleaned Data


In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)

clean_path = PROCESSED / "paperlane_orders_clean.csv"
complete_path = PROCESSED / "paperlane_orders_complete.csv"

df.to_csv(clean_path, index=False)
df_complete_sorted.to_csv(complete_path, index=False)

print("Saved:", clean_path.exists(), clean_path)
print("Saved:", complete_path.exists(), complete_path)


---
## ⭐ Optional Advanced Tasks — Solutions


In [ ]:
# Optional 1 — Revenue by Region
revenue_by_region = (
    df_complete
    .groupby("region", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        total_revenue=("revenue", "sum"),
        avg_order_value=("revenue", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)
revenue_by_region


In [ ]:
# Optional 2 — Revenue by Channel
revenue_by_channel = (
    df_complete
    .groupby("channel", as_index=False)
    .agg(
        orders=("order_id", "nunique"),
        total_revenue=("revenue", "sum"),
        avg_order_value=("revenue", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)
revenue_by_channel


In [ ]:
# Optional 3 — Pivot Table
pivot = pd.pivot_table(
    df_complete,
    values="revenue",
    index="region",
    columns="product",
    aggfunc="sum",
    fill_value=0
)
pivot
